In [ ]:
"""MARK: Optimized Colab Ollama Server"""

# @title 🛠️ Configuration
MODEL = "qwen3.5:27b" # @param {type:"string"}
PORT = "11434"

import os
import re
import time
import subprocess as sp
from pathlib import Path

def run_quiet(cmd):
    return sp.run(cmd, shell=True, capture_output=True, text=True)

# --- Step 1: Intelligent Installation ---
print("📦 Checking dependencies...", end=" ")
if not Path("/usr/bin/ollama").exists():
    run_quiet("curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr")

if not Path("/usr/local/bin/cloudflared").exists():
    run_quiet("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
    run_quiet("chmod +x /usr/local/bin/cloudflared")

run_quiet("pip install -q ollama")
print("Done.")

# --- Step 2: Service Management ---
print("🚀 Starting Ollama Service...", end=" ")
run_quiet(f"pkill -9 -f 'ollama serve'")

ollama_env = {
    **os.environ,
    "OLLAMA_HOST": f"0.0.0.0:{PORT}",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_ORIGINS": "*",
    "CUDA_VISIBLE_DEVICES": "0",
}

sp.Popen(["ollama", "serve"], env=ollama_env, stdout=sp.DEVNULL, stderr=sp.DEVNULL)

for _ in range(30):
    try:
        if run_quiet(f"curl -s http://127.0.0.1:{PORT}/api/tags").returncode == 0:
            break
    except: pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start")
print("Ready.")

# --- Step 3: Model Verification & Warmup ---
from ollama import Client
client = Client(host=f"http://127.0.0.1:{PORT}")

existing_models = [m.model for m in client.list().models]
if MODEL not in existing_models:
    print(f"📥 Pulling model: {MODEL}...")
    client.pull(MODEL)

print(f"🔥 Warming up {MODEL}...", end=" ", flush=True)
client.chat(model=MODEL, messages=[{"role": "user", "content": "hi"}], keep_alive=-1)
print("Optimized.")

# --- Step 4: Secure Tunneling ---
print("🌐 Establishing Cloudflare Tunnel...")
tunnel = sp.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=sp.PIPE, stderr=sp.STDOUT, text=True,
)

for line in tunnel.stdout:
    match = re.search(r"(https://[^\s]+\.trycloudflare\.com)", line)
    if match:
        url = match.group(1)
        print("\n" + "🟢" * 20)
        print(f"PUBLIC ENDPOINT: {url}")
        print("🟢" * 20 + "\n")
        break

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n👋 Shutting down...")
    tunnel.terminate()
    run_quiet("pkill -9 ollama")